In [22]:
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

import seaborn as sns
import matplotlib.pyplot as plt

import warnings
import numpy as np
warnings.filterwarnings('ignore')

In [24]:
df = pd.read_parquet("C:\\Langraph_MCP_SERVER_PRASTICS\\langraph\\data\\yellow_tripdata_2023-01.parquet")

In [25]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


## Question 1 :Read the data for January. How many columns are there?

In [26]:
answer=len(df.columns)

In [27]:
print('No of Columns :',answer)

No of Columns : 19


## Question 2:What's the standard deviation of the trips duration in January?

In [28]:
# 1. Convert columns to datetime (if they aren't already)
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

# 2. Filter for trips where pickup was in January
jan_df = df[df['tpep_pickup_datetime'].dt.month == 1].copy()

# 3. Calculate duration in minutes
jan_df['duration_minutes'] = (
    jan_df['tpep_dropoff_datetime'] - jan_df['tpep_pickup_datetime']
).dt.total_seconds() / 60

In [29]:
answer=jan_df['duration_minutes'].std()

In [30]:
answer

np.float64(42.585641764274165)

## Question 3 : What fraction of the records left after you dropped the outliers?

In [31]:
# 1. Filter for trips with duration between 1 and 60 minutes (inclusive)
jan_df_filtered = jan_df[
    (jan_df['duration_minutes'] >= 1) & (jan_df['duration_minutes'] <= 60)
]

# 2. Calculate the fraction of remaining records
fraction_left = len(jan_df_filtered) / len(jan_df)

print(f"Fraction of records left: {fraction_left:.4f}")
# Or as a percentage:
print(f"Percentage left: {fraction_left * 100:.2f}%")

Fraction of records left: 0.9812
Percentage left: 98.12%


In [36]:
df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
df['duration'] = df.duration.dt.total_seconds() / 60

In [37]:
df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

## Question 4 :What's the dimensionality of this matrix (number of columns)?

In [38]:
categorical = ['PULocationID', 'DOLocationID']
df[categorical] = df[categorical].astype(str)

In [39]:
train_dicts = df[categorical].to_dict(orient='records')

In [40]:
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

In [41]:
print(f'Feature matrix size: {X_train.shape}')

Feature matrix size: (3009173, 515)


## Question 5:What's the RMSE on train?

In [42]:
target = 'duration'
y_train = df[target].values

In [43]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)


In [44]:
print(f'Train RMSE: {np.sqrt(mean_squared_error(y_train, y_pred))}')

Train RMSE: 7.649261923332247


## Question 6:What's the RMSE on validation?

In [46]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].astype('str')
    
    return df

In [47]:
df_val = read_data("C:\\Langraph_MCP_SERVER_PRASTICS\\langraph\\data\\yellow_tripdata_2023-02.parquet")
val_dicts = df_val[categorical].to_dict(orient='records')
X_val = dv.transform(val_dicts) 
y_val = df_val.duration.values
y_pred = lr.predict(X_val)

In [49]:
print(f'Val RMSE: {np.sqrt(mean_squared_error(y_val, y_pred))}')

Val RMSE: 7.811816988184021
